## LLM as a judge

LLMs are not only used for conversational AI but also to evaluate the conversations. For example, Bavareso et al (2025) tested 11 LLMs against JUDGE-BENCH, a collection of 20 NLLP datasets with human annotations. LLMs are not specififically designed or trained to judge conversations but have some understanding of conversations and should be able to reflect on it. 

In this notebook, we will prompt an LLM to score agent responses for coherence and also to explain how it came to this score. We will use a bigger online LLM through the [ollama platform](https://ollama.com). Obviously, any other platform (OpenAI, Google) or local models can be used for the same purpose. 

We will load the conversations from the EMISSOR representations of conversations by loading pairs of user input and agent responses.

## Reference
Anna Bavaresco, Raffaella Bernardi, Leonardo Bertolazzi, Desmond Elliott, Raquel Fernández, Albert Gatt, Esam Ghaleb, Mario Giulianelli, Michael Hanna, Alexander Koller, Andre Martins, Philipp Mondorf, Vera Neplenbroek, Sandro Pezzelle, Barbara Plank, David Schlangen, Alessandro Suglia, Aditya K Surikuchi, Ece Takmaz, and Alberto Testoni. 2025. LLMs instead of Human Judges? A Large Scale Empirical Study across 20 NLP Evaluation Tasks. In Proceedings of the 63rd Annual Meeting of the Association for Computational Linguistics (Volume 2: Short Papers), pages 238–255, Vienna, Austria. Association for Computational Linguistics.

## Prerequisites
This notebook requires packages installed for running an ```ollama``` client and loading the EMISSOR scenarios. These packages can be installed through ```pip```:

```
!pip install ollama
!pip install emissor
```

In [1]:
import os
import re
from ollama import Client

The [ollama cloud service](https://ollama.com/cloud) requires an API key that can be obtained from the website: [https://docs.ollama.com/cloud#authentication](https://docs.ollama.com/cloud#authentication). Usage is still free but currently limited by hourly and weekly usage.

You need to obtain you own KEY for running this notebook.

## Defining an ollama cloud client

In [2]:
# OpenAI API Key
path = "../../ollama-cloud-key.txt"
api_key = "THIS SHOULD BE YOUR OLLAMA CLOUD API KEY"
with open(path) as f:
    api_key = f.read()

ollama_client = Client(host="https://ollama.com", headers={'Authorization': 'Bearer ' + api_key})

## Prompting for coherence

The next two functions define the prompt for evaluating the coherence of the agent responses to the user. The prompt is adapted to take into account that the Leolani agent is proactive in driving the conversation and can ask questions to the user as well.

In [3]:
def get_judge_prompt():
    instruction = """
                    ### Role Assignment
                    You are a Coherence Evaluation Judge.
                    Your job is to evaluate how coherent the **assistant’s response** is with respect to the **user’s input**.
                    The **assistant’s response** may be a follow up question to a statement from the **user**,  a related statement by the **agent** or the answer from the **agent** to a question of the **user**.
                    
                    ### Task Definition
                    You must:
                    1. Assign a **coherence score** from **0.0 to 1.0**
                    2. Provide a **short explanation** (maximum 2 sentences)
                    
                    ### Output Format (STRICT)
                    Return ONLY:
                    
                    <JSON>
                    {
                      "coherence_score": float between 0.0 and 1.0,
                      "explanation": "brief rationale"
                    }
                    </JSON>
                    """

    system_prompt = {
        "role": "system",
        "content": instruction
    }
    return system_prompt

def query_qwen_as_a_judge_ollama(messages, ollama_client, model):
    system_prompt = get_judge_prompt()
    messages_judge = [system_prompt] + messages
    response = ""
    for part in ollama_client.chat(model=model, messages=messages_judge, stream=True, format="json"):
        response += part['message']['content']
    # Look for patterns like "coherence_score": 0.9 or similar
    #     </analysis<|message|>The user said: "Lucy drinks wine". The assistant gave a garbled nonsense response. That's incoherent. Score low, maybe 0.0 or 0.1. Provide explanation.{
    #   "coherence_score": 0.0,
    #   "explanation": "The assistant's output is nonsensical and does not address the simple statement about Lucy drinking wine."
    #    }
    try:
        score_pattern = r'"coherence_score":\s*([\d.]+)'
        explanation_pattern = r'"explanation":\s*"([^"]+)"'

        score_match = re.search(score_pattern, response)
        explanation_match = re.search(explanation_pattern, response)

        if score_match and explanation_match:
            coherence_score = float(score_match.group(1))
            explanation = explanation_match.group(1)
            return {"coherence_score": coherence_score, "explanation": explanation}
        else:
            return {"coherence_score": None, "explanation": "Could not extract score and explanation from response"}
    except Exception as e:
        return {"coherence_score": None, "explanation": f"Error processing response: {str(e)}"}

## Obtaining the conversations from EMISSOR

The next two functions get the text signals from a conversation captured in an EMISSOR scenario and get the speaker of a text signal that represents a turn.

In [4]:
def get_text_signals_from_a_scenario(emissor_folder:str, scenario_id:str):
    text_signals=[]
    scenario_folder = os.path.join(emissor_folder, scenario_id)
    scenario_storage = ScenarioStorage(emissor_folder)
    scenario_ctrl = scenario_storage.load_scenario(scenario_id)
    try:
        text_signals = scenario_ctrl.get_signals(Modality.TEXT)
    except:
        print('Error loading text signals from text.json')
    return text_signals

def get_speaker_from_text_signal(textSignal: TextSignal):
    speaker = None
    mentions = textSignal.mentions
    for mention in mentions:
        annotations = mention.annotations
        for annotation in annotations:
            if annotation.type == 'ConversationalAgent':
                speaker = annotation.value
                break
        if speaker:
            break
    return speaker
    

## Scoring conversational pairs for coherence

The next code shows how the conversation from EMISSOR is processed as pairs of user input and agent responses and each of these is passed on the to LLM for assessing the coherence. It extracts a score for each pair and averages the score over all pairs.

In [18]:
from emissor.persistence import ScenarioStorage
from emissor.representation.scenario import Modality
from emissor.representation.scenario import Signal, TextSignal

EMISSOR="./data/emissor"
SCENARIO="1a093e5d-370d-479b-a2de-ed6e3f25538c"
SCENARIO="1d88aad3-959f-4daf-83a2-28d840b5be09"

In [19]:
text_signals = get_text_signals_from_a_scenario(EMISSOR, SCENARIO)
conversation = []
for text_signal in text_signals:
    signal_speaker = get_speaker_from_text_signal(text_signal) # SPEAKER or agent
    if signal_speaker == "SPEAKER":
        turn = {"role": "user", "content": text_signal.text}
    else:
        turn = {"role": "assistant", "content":  text_signal.text}
    conversation.append(turn)

In [20]:
model = "gpt-oss:120b"

coherence_scores = []
for turn in conversation[:9]:
    print(turn)
    if turn["role"] == "user":
        pair = [turn]
    else:
        pair.append(turn)
        coherence = query_qwen_as_a_judge_ollama(pair, ollama_client, model)
        print(coherence)
        if "coherence_score" in coherence:
            if not coherence["coherence_score"]==None:
                coherence_scores.append(coherence["coherence_score"])

average_coherence = sum(coherence_scores)/len(coherence_scores)
print("Average coherence", average_coherence)
print(coherence_scores)

{'role': 'assistant', 'content': "What's going on? Do you want to talk to me Yannis?"}
{'coherence_score': 0.1, 'explanation': "The user said goodbye, but the assistant responded with a mix of a farewell and unrelated questions, which does not directly address the user's intent."}
{'role': 'user', 'content': 'Yes'}
{'role': 'user', 'content': 'How are you?'}
{'role': 'assistant', 'content': "You see stranger I'm fine, thanks! What about you?"}
{'coherence_score': 0.85, 'explanation': 'The response directly answers the greeting and follows up with a reciprocal question, staying relevant despite minor grammatical oddities.'}
{'role': 'user', 'content': 'I am doing good.'}
{'role': 'assistant', 'content': 'I think you are right.'}
{'coherence_score': 0.0, 'explanation': "The response is nonsensical and does not address or relate to the user's statement about doing good."}
{'role': 'assistant', 'content': 'You have my confidence.'}
{'coherence_score': 0.2, 'explanation': "The assistant's r

Running the code again likely give you different results. It is wise to run the code several times and average over the different runs.

## End of notebook